# SHAMAN.OS Fine-Tune Pipeline
Run cells top to bottom. Set runtime to T4 GPU first (Runtime → Change runtime type).

In [ ]:
!pip install -q transformers datasets trl peft accelerate bitsandbytes sentencepiece protobuf huggingface_hub scipy numpy httpx gguf

In [ ]:
# HuggingFace login — required for Llama 3.2, skip if using TinyLlama
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Upload dataset from local machine
from google.colab import files
uploaded = files.upload()  # select dataset_combined.jsonl
DATASET_PATH = list(uploaded.keys())[0]
print('Dataset:', DATASET_PATH)

In [ ]:
# --- Step 0: Audit ---
import json, re, statistics
from dataclasses import dataclass

@dataclass
class AuditFlag:
    index: int
    check: str
    severity: str
    message: str
    detail: dict

EXPECTED_ROLES = ['system', 'user', 'assistant']
_LITERARY = ['the passage','the text','the author','the scripture','this section',
    'the narrator','the writing','describes','according to','the document',
    'in the book','the passage describes','as described']
_FIRST_PERSON = ['i ','i\'m','i\'ve','i can','i don\'t','i feel','i see',
    'i hear','i think','i am','my ','something is','there is','it\'s','it keeps']

def chk_structure(r, i):
    f = []
    if 'messages' not in r: return [AuditFlag(i,'structure','error','Missing messages key',{})]
    m = r['messages']
    if not isinstance(m, list): return [AuditFlag(i,'structure','error','messages not a list',{})]
    if len(m) != 3: return [AuditFlag(i,'structure','error',f'Expected 3 messages got {len(m)}',{})]
    for pos, item in enumerate(m):
        if not isinstance(item, dict): f.append(AuditFlag(i,'structure','error',f'msg {pos} not dict',{}))
        else:
            if 'role' not in item: f.append(AuditFlag(i,'structure','error',f'msg {pos} missing role',{}))
            if 'content' not in item: f.append(AuditFlag(i,'structure','error',f'msg {pos} missing content',{}))
    if f: return f
    roles = [x['role'] for x in m]
    if roles != EXPECTED_ROLES: f.append(AuditFlag(i,'structure','error',f'Wrong roles {roles}',{}))
    return f

def chk_empty(r, i):
    m = r.get('messages')
    if not isinstance(m, list) or len(m) != 3: return []
    return [AuditFlag(i,'empty_fields','warning',f'Empty content in {item.get("role",pos)}',{})
            for pos, item in enumerate(m) if isinstance(item,dict) and not item.get('content','').strip()]

def chk_literary(r, i):
    m = r.get('messages')
    if not isinstance(m, list) or len(m) < 2: return []
    c = m[1].get('content','').lower() if isinstance(m[1],dict) else ''
    found = [p for p in _LITERARY if p in c]
    return [AuditFlag(i,'literary_analysis','info','Literary language in user field',{'found':found})] if found else []

def chk_first_person(r, i):
    m = r.get('messages')
    if not isinstance(m, list) or len(m) < 2: return []
    c = m[1].get('content','').lower() if isinstance(m[1],dict) else ''
    return [] if any(p in c for p in _FIRST_PERSON) else [AuditFlag(i,'first_person','info','No first-person voice',{})]

def chk_seq_len(r, i, max_tok=512):
    m = r.get('messages')
    if not isinstance(m, list): return []
    text = ' '.join(x.get('content','') for x in m if isinstance(x,dict))
    est = len(text.split()) * 1.3
    return [AuditFlag(i,'sequence_length','warning',f'Est tokens {est:.0f} > {max_tok}',{})] if est > max_tok else []

def chk_dups(records, thr=0.70):
    def tok(r):
        try: return set(re.findall(r'\w+', r['messages'][1].get('content','').lower()))
        except: return set()
    sets = [tok(r) for r in records]
    flags = []
    for a in range(len(sets)):
        for b in range(a+1, len(sets)):
            d = max(len(sets[a]), len(sets[b]))
            if d and len(sets[a]&sets[b])/d > thr:
                flags.append(AuditFlag(b,'duplicate','warning',f'>{thr*100:.0f}% overlap with {a}',{'pair':[a,b]}))
    return flags

records, flags = [], []
with open(DATASET_PATH, encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line: continue
        try: records.append(json.loads(line))
        except Exception as e:
            flags.append(AuditFlag(i,'json_parse','error',str(e),{}))
            records.append({})

for i, r in enumerate(records):
    if not r: continue
    for fn in [chk_structure, chk_empty, chk_literary, chk_first_person, chk_seq_len]:
        flags.extend(fn(r, i))
flags.extend(chk_dups(records))

errors = sum(1 for f in flags if f.severity=='error')
print(f'Records: {len(records)} | Flags: {len(flags)} (errors={errors})')
print('READY' if errors == 0 else 'FIX ERRORS BEFORE TRAINING')

In [ ]:
# --- Step 1: Fine-tune with LoRA ---
import json, os, torch
from datetime import datetime, timezone
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

# Switch MODEL_ID to meta-llama/Llama-3.2-1B-Instruct once HF access is approved
MODEL_ID          = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR        = './shamanos_checkpoints'
FINAL_ADAPTER_DIR = './shamanos_adapter'
MAX_SEQ_LENGTH    = 512
LORA_RANK         = 8
LORA_ALPHA        = 16
BATCH_SIZE        = 4
GRAD_ACCUMULATION = 4
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 3
WARMUP_STEPS      = 10

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()

raw = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line: raw.append(json.loads(line))

ds = Dataset.from_list(raw).map(
    lambda ex: {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)}
)
split = ds.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')

bf16 = torch.cuda.is_bf16_supported() if device == 'cuda' else False
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        warmup_steps=WARMUP_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=(device=='cuda' and not bf16),
        bf16=bf16,
        logging_steps=5,
        eval_strategy='steps', eval_steps=25,
        save_strategy='steps', save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        report_to='none',
        dataloader_pin_memory=False,
        gradient_checkpointing=True,
        optim='adamw_torch',
        dataloader_num_workers=0,
        remove_unused_columns=False,
        dataset_text_field='text',
        max_length=MAX_SEQ_LENGTH,
        packing=False,
    ),
)

trainer.train()
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
with open(os.path.join(FINAL_ADAPTER_DIR, 'training_metadata.json'), 'w') as f:
    json.dump({'model_id': MODEL_ID, 'completed_at': datetime.now(timezone.utc).isoformat()}, f, indent=2)
print('Training complete. Adapter saved to:', FINAL_ADAPTER_DIR)

In [ ]:
# --- Step 2: Merge + Convert to GGUF Q4_K_M ---
import os, subprocess, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

ADAPTER_DIR  = './shamanos_adapter'
MERGED_DIR   = './shamanos_merged'
GGUF_F16     = './shamanos_1b.gguf'
GGUF_Q4      = './shamanos_1b_q4km.gguf'
LLAMACPP_DIR = './llama.cpp'

# 1. Merge adapter into base model
print('Merging adapter...')
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='cpu', low_cpu_mem_usage=True)
tok  = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=False, legacy=True)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR, torch_dtype=torch.float16).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
tok.save_pretrained(MERGED_DIR)

# Copy tokenizer.model if present in adapter dir (required by llama.cpp)
import shutil
for fname in ['tokenizer.model', 'tokenizer_config.json', 'special_tokens_map.json']:
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(MERGED_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Copied {fname} to merged dir')

print('Merge complete ->', MERGED_DIR)

# 2. Clone llama.cpp
if not os.path.exists(LLAMACPP_DIR):
    subprocess.run(['git','clone','https://github.com/ggerganov/llama.cpp', LLAMACPP_DIR], check=True)

# 3. Install llama.cpp python deps (gguf is required by convert_hf_to_gguf.py)
subprocess.run(['pip','install','-q','gguf','sentencepiece','numpy'], check=True)
req_file = f'{LLAMACPP_DIR}/requirements.txt'
if os.path.exists(req_file):
    subprocess.run(['pip','install','-q','-r', req_file], check=True)

# 4. Build llama-quantize
quantize_bin = f'{LLAMACPP_DIR}/build/bin/llama-quantize'
if not os.path.exists(quantize_bin):
    build_dir = f'{LLAMACPP_DIR}/build'
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(['cmake','..', '-DCMAKE_BUILD_TYPE=Release'], cwd=build_dir, check=True)
    subprocess.run(['cmake','--build','.','--config','Release','--target','llama-quantize','-j4'], cwd=build_dir, check=True)

# 5. Convert to GGUF f16
print('Converting to GGUF f16...')
result = subprocess.run(
    ['python3', f'{LLAMACPP_DIR}/convert_hf_to_gguf.py', MERGED_DIR, '--outfile', GGUF_F16, '--outtype', 'f16'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDOUT:', result.stdout[-3000:])
    print('STDERR:', result.stderr[-3000:])
    raise RuntimeError('GGUF conversion failed — see output above')
print('GGUF f16 saved:', GGUF_F16)

# 6. Quantize to Q4_K_M
qbin = quantize_bin if os.path.exists(quantize_bin) else f'{LLAMACPP_DIR}/llama-quantize'
subprocess.run([qbin, GGUF_F16, GGUF_Q4, 'Q4_K_M'], check=True)
print(f'Quantized GGUF: {GGUF_Q4} ({os.path.getsize(GGUF_Q4)/(1024**2):.0f} MB)')

In [ ]:
# Download finished GGUF to your local machine
from google.colab import files
files.download('./shamanos_1b_q4km.gguf')
files.download('./shamanos_adapter/training_metadata.json')